# 13 · Sealed internal test and blinded output assessment

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Run only after notebook 12. Initial inference generates independent rating jobs; a later cell joins actual adjudications without changing predictions.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Verify the original locked state

In [ ]:
from oncoplate.governance import verify_lock
lock=verify_lock(p['private']/'analysis_lock.json')
protocol=lock['protocol'];RUN_IDS=protocol['run_ids']
print('Locked primary contrast:',protocol['primary_contrast'],'Seeds:',protocol['model_seeds'])

## 2. Execute the frozen models and policies
This is the test-unblinding action. Predictions are saved BEFORE test claim ratings are imported.

In [ ]:
from oncoplate.inputs import make_partition_candidates
from oncoplate.evaluation import score_candidates,policy_predictions
import pandas as pd
frames=[]
for run_id in RUN_IDS:
    cc,objects,claim_dir=make_partition_candidates(cfg,run_id,'test',input_mode=protocol['input_mode'],asof=protocol['asof'],allow_unblind=True)
    sel=p['private']/'selectors'/run_id
    scored=score_candidates(cc,sel,read_json(sel/'chosen_support_calibrators.json'))
    seed=int(read_json(p['runs']/run_id/'run.json')['spec']['seed'])
    pred=policy_predictions(scored,read_json(sel/'operating_thresholds.json'),target=protocol['target_coverage'],seed=seed)
    write_table(claim_dir/'test_frozen_policy_predictions.csv',pred);frames.append(pred)
all_predictions=pd.concat(frames,ignore_index=True)
write_table(p['private']/'test_predictions_before_ratings.csv',all_predictions)
print('Blind rating jobs are in each run claim directory. Do not expose model scores to reviewers.')

## 3. Join independently adjudicated test ratings
Keep candidates with unknown references; do not delete hard cases or replace them with supported labels.

In [ ]:
from oncoplate.benchmark import adjudicated_ratings
from oncoplate.evaluation import attach_outcomes
frames=[]
for run_id in RUN_IDS:
    claim_dir=p['private']/'claims'/run_id/protocol['input_mode']
    pred=read_table(claim_dir/'test_frozen_policy_predictions.csv');pred['seed']=pred.seed.astype(int)
    ratings=adjudicated_ratings(read_table(claim_dir/'test_ratings_adjudicated.csv'))
    frames.append(attach_outcomes(pred,ratings))
results=pd.concat(frames,ignore_index=True)
write_table(p['private']/'test_evaluation_results.csv',results)
print('Results ready for notebook 14:',len(results))

## 4. External validation is a separate frozen-cohort addendum

In [ ]:
print('Use notebook 22 to ingest fresh external records without rebuilding or mutating the locked main target schema.')
print('Do not run notebook 03 again on the enlarged combined dataset after test lock.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
